In [33]:

from sklearn.datasets import fetch_openml
import numpy as np

mnist = fetch_openml('mnist_784', version=1)

X = mnist.data.values[:500] / 255.0   
y = mnist.target.astype(int).values[:500]


In [34]:



def one_hot(y, num_classes=10):
    oh = np.zeros((len(y), num_classes))
    oh[np.arange(len(y)), y] = 1
    return oh

y_onehot = one_hot(y)

In [36]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_onehot, test_size=0.2, random_state=42
)

In [37]:

def relu(x):
    return np.maximum(0, x)

def relu_deriv(x):
    return (x > 0).astype(float)

def softmax(x):
    exp = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp / np.sum(exp, axis=1, keepdims=True)


In [39]:


def cross_entropy(pred, y):
    return -np.mean(np.sum(y * np.log(pred + 1e-8), axis=1))


In [40]:

np.random.seed(42)
W1 = np.random.randn(784, 64) * 0.01
b1 = np.zeros((1, 64))

W2 = np.random.randn(64, 10) * 0.01
b2 = np.zeros((1, 10))

# -------------------------------
# Step 6: Forward Pass
# -------------------------------
def forward(X):
    z1 = X @ W1 + b1
    a1 = relu(z1)
    z2 = a1 @ W2 + b2
    a2 = softmax(z2)
    return z1, a1, z2, a2


In [41]:
# Step 7: Training Loop
# -------------------------------
def backward(X, y, z1, a1, a2, lr=0.01):
    global W1, b1, W2, b2
    m = X.shape[0]

    dz2 = a2 - y
    dW2 = a1.T @ dz2 / m
    db2 = np.sum(dz2, axis=0, keepdims=True) / m

    da1 = dz2 @ W2.T
    dz1 = da1 * relu_deriv(z1)
    dW1 = X.T @ dz1 / m
    db1 = np.sum(dz1, axis=0, keepdims=True) / m

    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2


In [42]:
epochs = 100
batch_size = 32
lr = 0.01

for epoch in range(epochs):
    # Shuffle training data
    perm = np.random.permutation(X_train.shape[0])
    X_train_shuffled = X_train[perm]
    y_train_shuffled = y_train[perm]

    # Mini-batch training
    for i in range(0, X_train.shape[0], batch_size):
        X_batch = X_train_shuffled[i:i+batch_size]
        y_batch = y_train_shuffled[i:i+batch_size]
        z1, a1, z2, a2 = forward(X_batch)
        backward(X_batch, y_batch, z1, a1, a2, lr)

    # Compute loss on full training set for monitoring
    _, _, _, a2_train = forward(X_train)
    train_loss = cross_entropy(a2_train, y_train)

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Training Loss: {train_loss:.4f}")

Epoch 0, Training Loss: 2.3011
Epoch 10, Training Loss: 2.2836
Epoch 20, Training Loss: 2.2358
Epoch 30, Training Loss: 2.1034
Epoch 40, Training Loss: 1.8648
Epoch 50, Training Loss: 1.5342
Epoch 60, Training Loss: 1.2014
Epoch 70, Training Loss: 0.9476
Epoch 80, Training Loss: 0.7681
Epoch 90, Training Loss: 0.6381


In [43]:
_, _, _, a2_test = forward(X_test)
y_pred = np.argmax(a2_test, axis=1)
y_true = np.argmax(y_test, axis=1)
test_acc = np.mean(y_pred == y_true)
print("Test Accuracy:", test_acc)

Test Accuracy: 0.77
